In [10]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/navyatripathi123/bank-marketing-practice01/bank-additional-full.csv
/kaggle/input/datasets/navyatripathi123/bank-marketing-practice01/bank-additional-names.txt
/kaggle/input/datasets/navyatripathi123/bank-marketing-practice01/bank-additional.csv
/kaggle/input/datasets/navyatripathi123/bank-marketing-practice01/bank-full.csv
/kaggle/input/datasets/navyatripathi123/bank-marketing-practice01/bank-names.txt
/kaggle/input/datasets/navyatripathi123/bank-marketing-practice01/bank.csv
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/__notebook__.ipynb
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/__output__.json
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/__results__.html
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/custom.css
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/X_test.csv
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/X_train.csv
/kaggle/input/notebooks/navyatripathi123/02-p

In [11]:
for root, dirs, files in os.walk("/kaggle/input"):
    if "X_train.csv" in files:
        print(root)

/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs


In [12]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs")

X_train = pd.read_csv(DATA_DIR / "X_train.csv")
X_val = pd.read_csv(DATA_DIR / "X_val.csv")
X_test = pd.read_csv(DATA_DIR / "X_test.csv")

y_train = pd.read_csv(DATA_DIR / "y_train.csv").squeeze()
y_val = pd.read_csv(DATA_DIR / "y_val.csv").squeeze()
y_test = pd.read_csv(DATA_DIR / "y_test.csv").squeeze()

In [13]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

print(y_train.shape)
print(y_val.shape)
print(y_test.shape)

(28831, 19)
(6178, 19)
(6179, 19)
(28831,)
(6178,)
(6179,)


In [14]:

# from pathlib import Path

# DATA_DIR = Path(
#     "/kaggle/input/datasets/navyatripathi123/bank-marketing-practice01"
# )

In [15]:
# df1 = pd.read_csv(DATA_DIR / "bank-full.csv", sep=";")
# df2 = pd.read_csv(DATA_DIR / "bank.csv", sep=";")
# df3 = pd.read_csv(DATA_DIR / "bank-additional-full.csv", sep=";")
# df4 = pd.read_csv(DATA_DIR / "bank-additional.csv", sep=";")

# PIPELINE

In [16]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# Identify numerical and categorical columns
cat_cols = X_train.select_dtypes(
    include=['object', 'category']
).columns.tolist()

num_cols = X_train.select_dtypes(
    include=np.number
).columns.tolist()

# Numerical preprocessing
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

# Categorical preprocessing
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(
        handle_unknown='ignore',
        drop='first'
    ))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# Fit on train; transform validation and test
X_train_encoded = preprocessor.fit_transform(X_train)
X_val_encoded = preprocessor.transform(X_val)
X_test_encoded = preprocessor.transform(X_test)

print("Train:", X_train_encoded.shape)
print("Validation:", X_val_encoded.shape)
print("Test:", X_test_encoded.shape)

Train: (28831, 52)
Validation: (6178, 52)
Test: (6179, 52)


In [17]:
# separate pipeline for lr due to its need of nuemeric feature scaling
from sklearn.preprocessing import StandardScaler

num_pipeline_logistic = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline_logistic = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(
        handle_unknown='ignore',
        drop='first'
    ))
])

preprocessor_logistic = ColumnTransformer([
    ('num', num_pipeline_logistic, num_cols),
    ('cat', cat_pipeline_logistic, cat_cols)
])

In [19]:
X_train_logistic = preprocessor_logistic.fit_transform(X_train)
X_val_logistic = preprocessor_logistic.transform(X_val)
X_test_logistic = preprocessor_logistic.transform(X_test)

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

logistic = LogisticRegression(
    max_iter=1000,
    random_state=42
)

decision_tree = DecisionTreeClassifier(
    random_state=42
)

random_forest = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1
)

In [21]:
logistic.fit(X_train_logistic, y_train)

decision_tree.fit(X_train_encoded, y_train)

random_forest.fit(X_train_encoded, y_train)

xgb.fit(X_train_encoded, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=-1,
              num_parallel_tree=None, ...)

In [23]:
logistic_prob = logistic.predict_proba(X_val_logistic)[:, 1]

dt_prob = decision_tree.predict_proba(X_val_encoded)[:, 1]

rf_prob = random_forest.predict_proba(X_val_encoded)[:, 1]

xgb_prob = xgb.predict_proba(X_val_encoded)[:, 1]

In [25]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import pandas as pd

models = {
    "Logistic Regression": (logistic, X_val_logistic),
    "Decision Tree": (decision_tree, X_val_encoded),
    "Random Forest": (random_forest, X_val_encoded),
    "XGBoost": (xgb, X_val_encoded)
}

results = []

for name, (model, X_val_model) in models.items():

    y_pred = model.predict(X_val_model)
    y_prob = model.predict_proba(X_val_model)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred),
        "Recall": recall_score(y_val, y_pred),
        "F1": f1_score(y_val, y_pred),
        "ROC-AUC": roc_auc_score(y_val, y_prob)
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "ROC-AUC",
    ascending=False
).reset_index(drop=True)

results_df.round(4)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,XGBoost,0.9011,0.6441,0.2730,0.3835,0.8069
1,Logistic Regression,0.9011,0.6906,0.2213,0.3351,0.8017
2,Random Forest,0.8951,0.5667,0.2931,0.3864,0.7828
3,Decision Tree,0.8338,0.3032,0.3664,0.3318,0.6311
